# Multi-Agent RAG POC - Feature 1: Query Expansion
This notebook verifies the Query Expansion Agent.

In [ ]:
import sys
import os
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from rag.ingest import load_pdf, chunk_text
from rag.embed import embed_texts
from rag.index import VectorStore
from orchestration.graph import app

### 1. Setup Data (Same as before)

In [ ]:
import fitz
sample_pdf_path = "../data/sample_paper.pdf"

# Checking if exists, otherwise create dummy
if not os.path.exists(sample_pdf_path):
    doc = fitz.open()
    page = doc.new_page()
    page.insert_text((50, 50), "Machine unlearning in federated learning involves removing a client's contribution without retraining. Exact unlearning is computationally expensive. Approximate unlearning uses gradient ascent.")
    doc.save(sample_pdf_path)
    print("Created dummy PDF")

In [ ]:
# Re-index to be sure
pages = load_pdf(sample_pdf_path)
chunks = chunk_text(pages)
chunk_texts = [c['page_content'] for c in chunks]
embeddings = embed_texts(chunk_texts)

vs = VectorStore(persist_directory="../data/chroma_db")
vs.add_documents(chunks, embeddings)

### 2. Verify Query Expansion Agent Isolated

In [ ]:
from agents.query_expansion import QueryExpansionAgent

agent = QueryExpansionAgent()
topic = "How to perform machine unlearning?"
queries = agent.process(topic)

print(f"Original: {topic}")
print("Expanded:")
for i, q in enumerate(queries):
    print(f"{i+1}. {q}")

### 3. Run Full Graph

In [ ]:
initial_state = {
    "query": "Methods for unlearning in FL",
    "expanded_queries": [],
    "retrieved_docs": [],
    "final_response": ""
}

# This will trigger the logging in retriever.py we added
result = app.invoke(initial_state)

print("FINAL SYNTHESIS:")
print(result['final_response'])